# Imports


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import torch

# Dataset


In [2]:
from datasets import load_dataset

d:\PyTorch\NLP\Sentiment Analysis with BERT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("stanfordnlp/imdb")

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## Train and Test dataset


In [5]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [6]:
train_dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [18]:
train_dataset.features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [19]:
for i in range(5):
    print(dataset["train"][i])
    print()

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

## Understand the labels


In [ ]:
train_dataset.features["label"]

ClassLabel(names=['neg', 'pos'])

## Check class balance


In [ ]:
from collections import Counter

Counter(train_dataset["label"])

Counter({0: 12500, 1: 12500})

In [ ]:
lengths = [len(text.split()) for text in train_dataset["text"]]

In [26]:
print(min(lengths))
print(max(lengths))
print(sum(lengths) / len(lengths))

10
2470
233.7872


We are going to use BERT model, that has window of 512 tokens. So, we can approximately provide it around 300 words.


# Tokenization


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

d:\PyTorch\NLP\Sentiment Analysis with BERT\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Fairuz\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
tokens = tokenizer(train_dataset[0]["text"])
print(tokens.keys())

KeysView({'input_ids': [101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107, 2004, 1996, 5148, 2162, 1998, 2679, 3314, 1999, 1996, 2142, 2163, 1012, 1999, 2090, 4851, 8801, 1998, 6623, 7939, 4697, 3619, 1997, 8947, 2055, 2037, 10740, 2006, 4331, 1010, 2016, 2038, 3348, 2007, 2014, 

The numbers are indexes in BERT's vocabulary.


In [ ]:
print(len(tokens["input_ids"]))

363


In [ ]:
decoded = tokenizer.convert_ids_to_tokens(tokens["input_ids"])

print(decoded)

['[CLS]', 'i', 'rented', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967', '.', 'i', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized', 'by', 'u', '.', 's', '.', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', ',', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', '"', 'controversial', '"', 'i', 'really', 'had', 'to', 'see', 'this', 'for', 'myself', '.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'the', 'plot', 'is', 'centered', 'around', 'a', 'young', 'swedish', 'drama', 'student', 'named', 'lena', 'who', 'wants', 'to', 'learn', 'everything', 'she', 'can', 'about', 'life', '.', 'in', 'particular', 'she', 'wants', 'to', 'focus', 'her', 'attention', '##s', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average', 'sw', '##ede', 'thought', 'about', 'certain', 'polit

In [ ]:
keys = tokens.keys()

In [46]:
import pandas as pd
import random
df = pd.DataFrame({
    "token_id": token_ids,
    "token": decoded_tokens
})

print(df.sample(10))

     token_id   token
116      2006      on
89       2315   named
34       2009      it
156      1010       ,
72       7987      br
242     25469  ##dity
199      2428  really
271      3152   films
290      2755    fact
261      2214     old


Bert has codes for all tthese words in own vocabs, it only sees the codes instead of the words.

the ## means : 
> This token is not a standalone word. It is attached to the previous token.

# Tokenize the whole dataset

Current dataset looks like:
```
{
 "text": "I loved this movie...",
 "label": 1
}
```
The model cannot use this.

BERT wants:
```
{
 "input_ids": [101, 1045, 2293, ...],
 "attention_mask": [1,1,1,...],
 "label": 1
}
```
So we are replacing: `text` with: `numbers representing text`

- Step 1: Create tokenizer function

We don't manually loop 50,000 times. Hugging Face datasets has .map().

Create:
```
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )
```

#### Arguments: 

`batch["text"]` : When .map() runs, it gives multiple examples at once.

Example: batch["text"]

might be:
```
[
 "I loved this movie",
 "This was terrible",
 "Amazing acting"
]
```

The tokenizer can process all of them.

`truncation=True` : If `review = 1000 tokens`, then first 512 tokens kept

`padding="max_length"` : If review = 200 tokens, then `200 tokens + 312 padding tokens`

so every sample becomes length 512.

>This is simple but inefficient.

A 20-token review becomes: 20 real tokens + 492 padding tokens

## Tokenizer Function

In [49]:
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length',max_length=512)

## Tokenize Dataset

In [50]:
tokenized_dataset=dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 50000/50000 [00:29<00:00, 1692.08 examples/s]


`batched=True` is important.

Without it: 
```
review 1 -> tokenize
review 2 -> tokenize
review 3 -> tokenize
```
With it:
```
batch of reviews
      ↓
tokenizer processes together
```
Much faster.

## Inspect the result

In [51]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [52]:
tokenized_dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [53]:
tokenized_dataset["train"][0].keys()

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [54]:
len(tokenized_dataset["train"][0]["input_ids"])

512

# Creating Model

                 ```
                 Text
                  |
                  ↓
             Tokenizer
                  |
                  ↓
          input_ids + attention_mask
                  |
                  ↓
              BERT model
        (already trained on language)
                  |
                  ↓
        [CLS] representation vector
                  |
                  ↓
          Classification layer
                  |
                  ↓
       Positive / Negative
```

## Load BERT

In [56]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2353.85it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [57]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

#### Check modle size

In [58]:
print(sum(p.numel() for p in model.parameters()))

109483778


#### Test a forward pass

In [60]:
sample = tokenized_dataset["train"][0]
sample.keys()

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

We only need input_ids and attention_mask

## Convert them to tensors

In [63]:
import torch 
input_ids=torch.tensor(
    sample['input_ids']
).unsqueeze(0)

attention_mask=torch.tensor(
    sample['attention_mask']
).unsqueeze(0)

unsqueeze(0): 

Because BERT expects batches.

It wants: (batch_size, sequence_length), Not: (sequence_length)

Before: [101, 1045, 2293, ...]

Shape: (512,)

After: [[101, 1045, 2293, ...]]

Shape: (1,512)

## Pass through BERT

In [64]:
output=model(
    input_ids=input_ids,
    attention_mask=attention_mask
)
print(output)

SequenceClassifierOutput(loss=None, logits=tensor([[-0.1912,  0.0490]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [65]:
output.logits

tensor([[-0.1912,  0.0490]], grad_fn=<AddmmBackward0>)

negative score = -0.1912

positive score = 0.0490

## Convert to prediction

In [66]:
prediction=torch.argmax(output.logits, dim=1)
print(prediction)

tensor([1])


In [67]:
print(output.logits)

tensor([[-0.1912,  0.0490]], grad_fn=<AddmmBackward0>)
